#**Task 2 : Concise Video Game Data Enhancement with Google AI Studio API**

## 1. SETUP

install library yang diperlukan seperti google-generativeai dan tqdm
1. google-generativeai
  digunakan untuk:
  - Mengakses model AI generatif Gemini dari Google
  - Mengirim prompt (teks perintah) ke model
  - Mendapatkan jawaban otomatis dari model AI
2. tqdm
  digunakan untuk: Menampilkan progress bar saat loop

In [2]:
pip install google-generativeai

In [3]:
!pip install -q tqdm

## 2. Google API Key

- Mengimpor library getpass untuk meminta pengguna untuk memasukkan API key tanpa ditampilkan agar aman saat script ini dibagikan. Karena API key tidak boleh dishare ke orang lain karena berisi data dari pengguna
- Mengimpor library resmi Google Gemini API (google-generativeai) untuk mengatur konfigurasi API key agar bisa memakai layanan Gemini.

In [1]:
import getpass
import google.generativeai as genai

# Minta pengguna memasukkan API key tanpa ditampilkan
API_KEY = getpass.getpass("Masukkan API Key dari https://aistudio.google.com/app/apikey: ")

# Konfigurasi Gemini API
genai.configure(api_key=API_KEY)

Masukkan API Key dari https://aistudio.google.com/app/apikey: ··········


##3. READ DATA

Mengupload datset yang digunakan menggunakan library files dari google colab dan membaca data dengan library pandas

In [2]:
from google.colab import files

# Upload file CSV, misalnya: game_data.csv
uploaded = files.upload()

Saving Game Thumbnail.csv to Game Thumbnail.csv


In [3]:
import pandas as pd

# Ganti nama file sesuai file yang kamu upload
df = pd.read_csv("Game Thumbnail.csv")

# Lihat 5 baris pertama untuk konfirmasi
df.head()

,game_title,image_url
0,Street Fighter 6,https://images.igdb.com/igdb/image/upload/t_co...
1,Hunt: Showdown 1896,https://images.igdb.com/igdb/image/upload/t_co...
2,Wuthering Waves,https://images.igdb.com/igdb/image/upload/t_co...
3,Arma Reforger,https://images.igdb.com/igdb/image/upload/t_co...
4,Lethal Company,https://images.igdb.com/igdb/image/upload/t_co...


## 4. Menambahkan kolom dan mengisi kolom dengan menggunakan hasil prompt dari gemini

Membuat fungsi generate_text() yang digunakan untuk mengirim prompt ke model AI Google Gemini dan mengembalikan hasil teks dari AI tersebut. Fungsi ini dilengkapi dengan mekanisme retry otomatis untuk mengantisipasi kegagalan saat memanggil API. Di dalamnya, model gemini-2.0-flash-lite digunakan karena versi ini ringan dan cepat. Jika permintaan gagal, fungsi akan mencoba ulang hingga tiga kali dengan jeda dua detik antar percobaan. Jika semua percobaan gagal, fungsi akan mengembalikan nilai "N/A". Dengan pendekatan ini, fungsi generate_text() menjadi lebih stabil dan tahan terhadap gangguan jaringan atau pembatasan sementara dari API.

In [4]:
import time
import re
from tqdm import tqdm

def generate_text(prompt, retries=3, delay=2):
    for attempt in range(retries):
        try:
            model = genai.GenerativeModel("gemini-2.0-flash-lite")
            response = model.generate_content(prompt)
            return response.text.strip()
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(delay)
    return "N/A"

### 4.1 Kolom Genre

Melakukan proses pengambilan genre utama dari setiap judul game dalam DataFrame df. Pertama, genre_list diinisialisasi sebagai list kosong untuk menyimpan hasil. Lalu, dengan bantuan tqdm, dilakukan iterasi untuk menampilkan progress bar saat memproses kolom game_title. Untuk setiap judul game (title), dibentuk sebuah prompt yang meminta AI menyebutkan genre utama dalam satu kata, contohnya seperti "Aksi", "Shooter", atau "Strategi". Prompt ini dikirim ke fungsi generate_text() yang akan mengembalikan respons dari model AI. Kemudian hasil mentah (genre_raw) dibersihkan menggunakan regular expression re.findall(r'\w+', genre_raw), yang akan mengekstrak hanya huruf dan angka (menghilangkan karakter spesial atau tanda baca). Jika hasil bersih (genre_clean) ada isinya, maka hanya kata pertama yang diambil sebagai genre, jika tidak ada, maka akan diisi dengan string default "Lainnya". Genre akhir ini lalu ditambahkan ke dalam genre_list.

In [5]:
genre_list = []

for title in tqdm(df["game_title"]):
    prompt_genre = (
        f"Apa genre utama dari video game berjudul '{title}'? "
        f"Jawab dengan satu kata saja seperti: Aksi, Petualangan, Shooter, Strategi, dll."
    )

    genre_raw = generate_text(prompt_genre)
    genre_clean = re.findall(r'\w+', genre_raw)
    genre = genre_clean[0] if genre_clean else "Lainnya"
    genre_list.append(genre)


 10%|█         | 5/50 [00:21<02:17,  3.05s/it]ERROR:tornado.access:500 POST /v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 13960.23ms


Attempt 1 failed: 500 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: TypeError: Failed to fetch


 16%|█▌        | 8/50 [00:42<03:09,  4.51s/it]ERROR:tornado.access:500 POST /v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 12347.75ms


Attempt 1 failed: 500 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: TypeError: Failed to fetch


100%|██████████| 50/50 [04:10<00:00,  5.02s/it]


In [6]:
df["genre"] = genre_list

# Lihat hasilnya
df

,game_title,image_url,genre
0,Street Fighter 6,https://images.igdb.com/igdb/image/upload/t_co...,Pertarungan
1,Hunt: Showdown 1896,https://images.igdb.com/igdb/image/upload/t_co...,Shooter
2,Wuthering Waves,https://images.igdb.com/igdb/image/upload/t_co...,Aksi
3,Arma Reforger,https://images.igdb.com/igdb/image/upload/t_co...,Simulasi
4,Lethal Company,https://images.igdb.com/igdb/image/upload/t_co...,Horor
5,Arena Breakout: Infinite,https://images.igdb.com/igdb/image/upload/t_co...,Shooter
6,Zenless Zone Zero,https://images.igdb.com/igdb/image/upload/t_co...,Aksi
7,ARK: Survival Ascended,https://images.igdb.com/igdb/image/upload/t_co...,Bertahan
8,Sid Meier's Civilization VII,https://images.igdb.com/igdb/image/upload/t_co...,Strategi
9,SMITE 2,https://images.igdb.com/igdb/image/upload/t_co...,MOBA


### 4.2 Kolom Short Description

Pertama, dibuat list kosong bernama desc_list yang akan menyimpan hasil deskripsi tiap game. Lalu dilakukan iterasi terhadap kolom game_title menggunakan tqdm agar muncul progress bar. Untuk setiap title, disusun prompt yang meminta AI untuk membuat deskripsi singkat kurang dari 30 kata yang mencerminkan gameplay dan konsep utama dari game tersebut. Hasil dari prompt disimpan dalam desc_raw, lalu dibersihkan menggunakan re.findall(r'.{1,30}', desc_raw). Namun, perlu dicatat bahwa ekspresi ini justru memotong deskripsi setiap 30 karakter (bukan kata), yang mungkin tidak sesuai dengan maksud awal. Hasil potongan tersebut kemudian disatukan kembali menjadi satu string desc, atau diisi "Deskripsi tidak tersedia" jika kosong. Terakhir, deskripsi itu dimasukkan ke dalam desc_list.

In [7]:
desc_list = []

# Loop untuk setiap judul game dan mengambil deskripsi
for title in tqdm(df["game_title"]):
    # Prompt deskripsi singkat
    prompt_desc = (
        f"Tuliskan deskripsi singkat di bawah 30 kata untuk game berjudul '{title}'. "
        f"Pastikan deskripsi menggambarkan gameplay dan konsep utama dari game tersebut."
    )

    # Ambil hasil deskripsi
    desc_raw = generate_text(prompt_desc)
    desc_clean = re.findall(r'.{1,30}', desc_raw)  # ambil kata yang valid dan pastikan tidak lebih dari 30 kata
    desc = ' '.join(desc_clean) if desc_clean else "Deskripsi tidak tersedia"
    desc_list.append(desc)

  4%|▍         | 2/50 [00:48<21:18, 26.63s/it]ERROR:tornado.access:500 POST /v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 482.30ms


Attempt 1 failed: 500 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: TypeError: Failed to fetch


ERROR:tornado.access:500 POST /v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 31623.20ms


Attempt 2 failed: 500 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: TypeError: Failed to fetch


 26%|██▌       | 13/50 [02:27<06:13, 10.10s/it]ERROR:tornado.access:500 POST /v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 13986.33ms


Attempt 1 failed: 500 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: TypeError: Failed to fetch


 88%|████████▊ | 44/50 [06:11<00:37,  6.29s/it]ERROR:tornado.access:500 POST /v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 11234.79ms


Attempt 1 failed: 500 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: TypeError: Failed to fetch


100%|██████████| 50/50 [06:53<00:00,  8.27s/it]


In [8]:
df["short_description"] = desc_list

# Lihat hasilnya
df

,game_title,image_url,genre,short_description
0,Street Fighter 6,https://images.igdb.com/igdb/image/upload/t_co...,Pertarungan,Street Fighter 6: Pertarungan jalanan klasik ...
1,Hunt: Showdown 1896,https://images.igdb.com/igdb/image/upload/t_co...,Shooter,Berburu monster brutal dan pem ain lain dalam ...
2,Wuthering Waves,https://images.igdb.com/igdb/image/upload/t_co...,Aksi,Wuthering Waves adalah RPG aks i dunia terbuka...
3,Arma Reforger,https://images.igdb.com/igdb/image/upload/t_co...,Simulasi,Arma Reforger adalah game simu lasi perang mul...
4,Lethal Company,https://images.igdb.com/igdb/image/upload/t_co...,Horor,Lethal Company adalah game hor or kooperatif t...
5,Arena Breakout: Infinite,https://images.igdb.com/igdb/image/upload/t_co...,Shooter,"Masuki Arena Breakout: Infinit e, penembak FPS..."
6,Zenless Zone Zero,https://images.igdb.com/igdb/image/upload/t_co...,Aksi,"""Zenless Zone Zero"" adalah gam e aksi RPG yang..."
7,ARK: Survival Ascended,https://images.igdb.com/igdb/image/upload/t_co...,Bertahan,Bertahan hidup di dunia purba yang liar! Kump...
8,Sid Meier's Civilization VII,https://images.igdb.com/igdb/image/upload/t_co...,Strategi,Raih kejayaan dalam Civilizati on VII! Bangun ...
9,SMITE 2,https://images.igdb.com/igdb/image/upload/t_co...,MOBA,SMITE 2 adalah MOBA dewa-dewi generasi beriku...


### 4.3 Kolom Player Mode

Menambahkan kolom baru yang berisi informasi tentang mode permainan setiap game dalam DataFrame df. Untuk setiap judul game di kolom game_title, kode membuat prompt untuk mengidentifikasi apakah game dimainkan dalam mode "Pemain Tunggal", "Pemain Banyak", atau "Keduanya". Model menghasilkan jawaban yang kemudian dibersihkan dan divalidasi dengan memeriksa kata-kata kunci dalam hasilnya. Berdasarkan hasil ini, mode permainan dikategorikan dan ditambahkan ke dalam list mode_list. Setelah semua game diproses, kolom baru player_mode yang berisi informasi mode permainan dimasukkan ke dalam DataFrame.

In [9]:
mode_list = []

# Loop untuk setiap judul game dan mengambil mode permainan
for title in tqdm(df["game_title"]):
    # Prompt mode permainan
    prompt_mode = (
        f"Game berjudul '{title}' dimainkan dalam mode apa? "
        f"Pilih satu jawaban: Pemain Tunggal, Pemain Banyak, atau Keduanya."
    )

    mode_raw = generate_text(prompt_mode)

    # Validasi hasil mode
    mode_clean = mode_raw.lower()
    if "tunggal" in mode_clean and "banyak" not in mode_clean:
        mode = "Pemain Tunggal"
    elif "banyak" in mode_clean and "tunggal" not in mode_clean:
        mode = "Pemain Banyak"
    elif "tunggal" in mode_clean and "banyak" in mode_clean or "keduanya" in mode_clean:
        mode = "Keduanya"
    else:
        mode = "Tidak Diketahui"

    mode_list.append(mode)

# Gabungkan ke DataFrame sebagai kolom baru
df["player_mode"] = mode_list

 30%|███       | 15/50 [00:55<03:28,  5.96s/it]ERROR:tornado.access:500 POST /v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 10926.22ms


Attempt 1 failed: 500 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: TypeError: Failed to fetch


100%|██████████| 50/50 [04:41<00:00,  5.62s/it]


In [10]:
df

,game_title,image_url,genre,short_description,player_mode
0,Street Fighter 6,https://images.igdb.com/igdb/image/upload/t_co...,Pertarungan,Street Fighter 6: Pertarungan jalanan klasik ...,Keduanya
1,Hunt: Showdown 1896,https://images.igdb.com/igdb/image/upload/t_co...,Shooter,Berburu monster brutal dan pem ain lain dalam ...,Keduanya
2,Wuthering Waves,https://images.igdb.com/igdb/image/upload/t_co...,Aksi,Wuthering Waves adalah RPG aks i dunia terbuka...,Keduanya
3,Arma Reforger,https://images.igdb.com/igdb/image/upload/t_co...,Simulasi,Arma Reforger adalah game simu lasi perang mul...,Keduanya
4,Lethal Company,https://images.igdb.com/igdb/image/upload/t_co...,Horor,Lethal Company adalah game hor or kooperatif t...,Pemain Banyak
5,Arena Breakout: Infinite,https://images.igdb.com/igdb/image/upload/t_co...,Shooter,"Masuki Arena Breakout: Infinit e, penembak FPS...",Pemain Banyak
6,Zenless Zone Zero,https://images.igdb.com/igdb/image/upload/t_co...,Aksi,"""Zenless Zone Zero"" adalah gam e aksi RPG yang...",Pemain Tunggal
7,ARK: Survival Ascended,https://images.igdb.com/igdb/image/upload/t_co...,Bertahan,Bertahan hidup di dunia purba yang liar! Kump...,Keduanya
8,Sid Meier's Civilization VII,https://images.igdb.com/igdb/image/upload/t_co...,Strategi,Raih kejayaan dalam Civilizati on VII! Bangun ...,Keduanya
9,SMITE 2,https://images.igdb.com/igdb/image/upload/t_co...,MOBA,SMITE 2 adalah MOBA dewa-dewi generasi beriku...,Pemain Banyak


## 5. Menyimpan dan download data yang sudah di tambahkan dengan format CSV

In [11]:
df.to_csv("game_data_enriched.csv", index=False)

# Unduh file hasilnya
files.download("game_data_enriched.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [195]:
import pandas as pd

# Ganti nama file sesuai file yang kamu upload
df = pd.read_csv("game_data_enriched.csv")

# Lihat 5 baris pertama untuk konfirmasi
df

,game_title,image_url,genre,desc,mode_permainan
0,Street Fighter 6,https://images.igdb.com/igdb/image/upload/t_co...,Pertarungan,Street Fighter 6: Hadirkan per tempuran jalana...,Keduanya
1,Hunt: Showdown 1896,https://images.igdb.com/igdb/image/upload/t_co...,Shooter,Berburu iblis di rawa Louisian a yang gelap. B...,Keduanya
2,Wuthering Waves,https://images.igdb.com/igdb/image/upload/t_co...,Aksi,Jelajahi dunia terbuka luas ya ng dilanda benc...,Keduanya
3,Arma Reforger,https://images.igdb.com/igdb/image/upload/t_co...,Shooter,'Arma Reforger' adalah game mi liter multiplay...,Keduanya
4,Lethal Company,https://images.igdb.com/igdb/image/upload/t_co...,Horor,Bertahan hidup sebagai pekerja kontrak korpor...,Pemain Banyak
5,Arena Breakout: Infinite,https://images.igdb.com/igdb/image/upload/t_co...,Shooter,"Masuk ke dunia Arena Breakout: Infinite, game...",Pemain Banyak
6,Zenless Zone Zero,https://images.igdb.com/igdb/image/upload/t_co...,Aksi,'Zenless Zone Zero' adalah RPG aksi cepat di ...,Keduanya
7,ARK: Survival Ascended,https://images.igdb.com/igdb/image/upload/t_co...,Bertahan,Bertahan hidup di dunia prasej arah! Jelajahi ...,Keduanya
8,Sid Meier's Civilization VII,https://images.igdb.com/igdb/image/upload/t_co...,Strategi,Bangun peradaban melalui waktu ! Kembangkan ko...,Keduanya
9,SMITE 2,https://images.igdb.com/igdb/image/upload/t_co...,MOBA,"SMITE 2 adalah MOBA dewa-dewi, bertempur dala...",Pemain Banyak
